In [ ]:
# Install required packages
!pip install scikit-learn joblib pandas

# --- Imports ---
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib
from google.colab import files


df = pd.read_csv("dataset.csv")

# --- Clean & Preprocess ---
df.dropna(subset=['Description', 'Severity', 'CVSS_Score', 'Attack_Vector'], inplace=True)
df['Severity'] = df['Severity'].str.capitalize()

# --- Features & Target ---
X_text = df['Description']
X_num = df[['CVSS_Score']]
X_cat = df[['Attack_Vector']]
y = df['Severity']

# Combine all features
X_all = pd.concat([X_text, X_num, X_cat], axis=1)

# --- Preprocessing Pipelines ---
text_transformer = TfidfVectorizer(max_features=1000)
num_transformer = StandardScaler()
cat_transformer = OneHotEncoder(handle_unknown='ignore')

# Create column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('text', text_transformer, 'Description'),
        ('num', num_transformer, ['CVSS_Score']),
        ('cat', cat_transformer, ['Attack_Vector'])
    ]
)

# --- Model Pipeline ---
model = make_pipeline(
    preprocessor,
    RandomForestClassifier(n_estimators=100, random_state=42)
)

# --- Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42)

# --- Train ---
model.fit(X_train, y_train)

# --- Evaluate ---
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

# --- Save Model ---
joblib.dump(model, 'vulnerability_model.pkl')
files.download('vulnerability_model.pkl')


              precision    recall  f1-score   support

    Critical       0.26      0.26      0.26      4037
        High       0.26      0.26      0.26      4003
         Low       0.26      0.25      0.25      3983
      Medium       0.24      0.24      0.24      4007

    accuracy                           0.25     16030
   macro avg       0.25      0.25      0.25     16030
weighted avg       0.25      0.25      0.25     16030



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>